In [2]:
import kagglehub
import pandas as pd
import os

path = kagglehub.dataset_download("hosammhmdali/house-price-dataset")
print("Path to dataset files:", path)

# Buscamos el archivo CSV dentro de la ruta de descarga
# (Suele llamarse data.csv, house_prices.csv, etc. Puedes mirar el nombre con la Opción 1)
archivo_csv = os.path.join(path, "housing.csv") 

# Cargamos el dataset
df = pd.read_csv(archivo_csv)

# Mostramos las primeras filas
df.head()

Path to dataset files: /Users/felipe/.cache/kagglehub/datasets/hosammhmdali/house-price-dataset/versions/1


,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,median_house_value,ocean_proximity
0,-122.23,37.88,41.0,880.0,129.0,322.0,126.0,8.3252,452600.0,NEAR BAY
1,-122.22,37.86,21.0,7099.0,1106.0,2401.0,1138.0,8.3014,358500.0,NEAR BAY
2,-122.24,37.85,52.0,1467.0,190.0,496.0,177.0,7.2574,352100.0,NEAR BAY
3,-122.25,37.85,52.0,1274.0,235.0,558.0,219.0,5.6431,341300.0,NEAR BAY
4,-122.25,37.85,52.0,1627.0,280.0,565.0,259.0,3.8462,342200.0,NEAR BAY


In [4]:
import ee
import os
import requests
import pandas as pd

# 1. Inicializar com o teu projeto
ee.Initialize(project="pklot-termposter") 

# 2. Pasta de saída
OUTPUT_DIR = "imagenes_satelite"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# 3. Config
START_DATE = "2010-01-01"        # busca NAIP desde esta data até hoje
BUFFER_METROS = 100              # ~200m x 200m ao redor do ponto
DIMENSIONS = "1024x1024"         # resolução da imagem em pixels
FORCE_REDOWNLOAD = False         # True para baixar de novo mesmo se já existir
SOLO_MAS_RECIENTE = True         # True = só a imagem mais recente por ponto
                                 # False = todas as datas disponíveis

# 4. ID único e coluna de rutas
df['id_casa'] = range(1, len(df) + 1)
df['ruta_imagen'] = ""

# 5. Função de download usando NAIP
def descargar_naip(lat, lon, id_casa):
    punto = ee.Geometry.Point([lon, lat])
    region = punto.buffer(BUFFER_METROS).bounds()

    coleccion = (ee.ImageCollection("USDA/NAIP/DOQQ")
                 .filterBounds(punto)
                 .filterDate(START_DATE, "2026-01-01")
                 .sort("system:time_start"))

    n = coleccion.size().getInfo()
    if n == 0:
        print(f"  ⚠️  Sin imágenes NAIP para casa ID {id_casa} (¿fuera de EE.UU.?)")
        return []

    # Decide quais imagens baixar
    if SOLO_MAS_RECIENTE:
        imagenes = [ee.Image(coleccion.sort("system:time_start", False).first())]
    else:
        lista = coleccion.toList(n)
        imagenes = [ee.Image(lista.get(i)) for i in range(n)]

    rutas = []
    for img in imagenes:
        date = img.date().format("YYYY-MM-dd").getInfo()
        nombre_archivo = os.path.join(OUTPUT_DIR, f"casa_{id_casa}_{date}.png")

        if os.path.exists(nombre_archivo) and not FORCE_REDOWNLOAD:
            print(f"  {date} - ya existe, saltando")
            rutas.append(nombre_archivo)
            continue

        url = img.select(["R", "G", "B"]).getThumbURL({
            "region": region,
            "dimensions": DIMENSIONS,
            "format": "png"
        })

        r = requests.get(url)
        if r.status_code == 200:
            with open(nombre_archivo, "wb") as f:
                f.write(r.content)
            print(f"  📸 Casa ID {id_casa} ({date}) → {nombre_archivo}")
            rutas.append(nombre_archivo)
        else:
            print(f"  ❌ Error {r.status_code} en casa ID {id_casa} ({date})")

    return rutas

# 6. Descargar las primeras 5 como prueba
print("Iniciando descarga con NAIP...\n")
for indice, fila in df.head(500).iterrows():
    id_actual = fila['id_casa']
    print(f"=== Casa ID {id_actual} ===")
    rutas = descargar_naip(fila['latitude'], fila['longitude'], id_casa=id_actual)

    if rutas:
        # Guarda la ruta más reciente (o la única) en el DataFrame
        df.at[indice, 'ruta_imagen'] = rutas[-1]

print("\n¡Proceso terminado!")
df[['id_casa', 'latitude', 'longitude', 'median_house_value', 'ruta_imagen']].head()

Iniciando descarga con NAIP...

=== Casa ID 1 ===
  2022-05-18 - ya existe, saltando
=== Casa ID 2 ===
  2022-05-18 - ya existe, saltando
=== Casa ID 3 ===
  2022-05-18 - ya existe, saltando
=== Casa ID 4 ===
  2022-05-18 - ya existe, saltando
=== Casa ID 5 ===
  2022-05-18 - ya existe, saltando
=== Casa ID 6 ===
  📸 Casa ID 6 (2022-05-18) → imagenes_satelite/casa_6_2022-05-18.png
=== Casa ID 7 ===
  📸 Casa ID 7 (2022-05-18) → imagenes_satelite/casa_7_2022-05-18.png
=== Casa ID 8 ===
  📸 Casa ID 8 (2022-05-18) → imagenes_satelite/casa_8_2022-05-18.png
=== Casa ID 9 ===
  📸 Casa ID 9 (2022-05-18) → imagenes_satelite/casa_9_2022-05-18.png
=== Casa ID 10 ===
  📸 Casa ID 10 (2022-05-18) → imagenes_satelite/casa_10_2022-05-18.png
=== Casa ID 11 ===
  📸 Casa ID 11 (2022-05-18) → imagenes_satelite/casa_11_2022-05-18.png
=== Casa ID 12 ===
  📸 Casa ID 12 (2022-05-18) → imagenes_satelite/casa_12_2022-05-18.png
=== Casa ID 13 ===
  📸 Casa ID 13 (2022-05-18) → imagenes_satelite/casa_13_2022-05-18

,id_casa,latitude,longitude,median_house_value,ruta_imagen
0,1,37.88,-122.23,452600.0,imagenes_satelite/casa_1_2022-05-18.png
1,2,37.86,-122.22,358500.0,imagenes_satelite/casa_2_2022-05-18.png
2,3,37.85,-122.24,352100.0,imagenes_satelite/casa_3_2022-05-18.png
3,4,37.85,-122.25,341300.0,imagenes_satelite/casa_4_2022-05-18.png
4,5,37.85,-122.25,342200.0,imagenes_satelite/casa_5_2022-05-18.png
